# Paper 2 demo: nonconvex Ball-DP transcript releases

This notebook is the one-example companion to `paper2_nonconvex_transcript_ball_dp.tex`.
It uses the theorem-backed operator-norm one-hidden-layer tanh network and follows the paper's transcript view of DP-SGD.

The release at step $t$ is the sanitized clipped gradient aggregate
\[
\widetilde S_t = \sum_{i\in B_t} \mathrm{clip}_{C}(\nabla_\theta \ell(\theta_t;z_i)) + \xi_t,
\qquad \xi_t\sim\mathcal N(0,\nu_t^2 I).
\]
For same-label embedding perturbations inside radius $r$, the certified per-step replacement signal is
\[
\Delta_t(r) \le \min\{L_z r, 2C\}.
\]
The Paper 2 bound evaluates the direct Hayes-style product reference experiment, exposed in the code as `ball_rero(..., mode="hayes")`.


In [ ]:
from __future__ import annotations

from types import SimpleNamespace
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jax.random as jr

from quantbayes.ball_dp import (
    ArrayDataset,
    ball_rero,
    make_finite_identification_prior,
    make_trace_metadata_from_release,
    summarize_embedding_ball_radii,
)
from quantbayes.ball_dp.attacks import BallTraceMapAttackConfig
from quantbayes.ball_dp.attacks.gradient_based import subtract_known_batch_gradients
from quantbayes.ball_dp.experiments.run_attack_experiment import (
    DEFAULT_ORDERS,
    find_feasible_support_banks,
    load_embeddings,
    make_support_set,
    radius_value_from_report,
    remove_train_index,
    resolve_dataset,
)
from quantbayes.ball_dp.experiments.run_nonconvex_transcript_experiment import (
    calibrate_noise,
    compute_hayes_bounds,
    resolve_task,
    stratified_subsample,
)
from quantbayes.ball_dp.experiments.run_nonconvex_transcript_attack_experiment import (
    train_recorded_release,
    run_one_attack,
)
from quantbayes.ball_dp.theorem import (
    TheoremBounds,
    TheoremModelSpec,
    TrainConfig,
    certified_constants,
    certified_lz,
    fit_release,
    make_model,
)


## Configuration

The defaults are deliberately small so the notebook is usable as a demo.  The official scripts use the same logic but sweep datasets, privacy targets, seeds, and supports.


In [ ]:
DATASET = "MNIST-embeddings"
MAX_TRAIN = 1024
MAX_TEST = 512
SUBSAMPLE_SEED = 0

HIDDEN_DIM = 128
A = 4.0
LAMBDA = 4.0
INPUT_BOUND_MARGIN = 1.001

RADIUS_TAG = "q80"
EPSILON = 8.0
DELTA = 1e-6
M = 8

NUM_STEPS = 100
BATCH_SIZE = 128
CLIP_NORM = 1.0
LEARNING_RATE = 3e-3
EVAL_EVERY = 50
RELEASE_SEED = 0


## Load embeddings and choose the policy radius

The metric is label-preserving Euclidean distance: records with different labels are at distance $\infty$, and same-label records are compared in embedding space.


In [ ]:
loader_args = SimpleNamespace(
    dataset=DATASET,
    data_root="data",
    embedding_batch_size=256,
    allow_cpu_fallback=True,
    embedding_cache_path=None,
    force_recompute_embeddings=False,
    num_workers=2,
    hf_cache_dir=None,
    encoder_model_name="sentence-transformers/all-MiniLM-L6-v2",
    max_length=256,
)

dataset_spec = resolve_dataset(DATASET)
loaded = load_embeddings(loader_args, dataset_spec)
X_train, y_train = stratified_subsample(
    loaded.X_train, loaded.y_train, max_examples=MAX_TRAIN, seed=SUBSAMPLE_SEED
)
X_test, y_test = stratified_subsample(
    loaded.X_test, loaded.y_test, max_examples=MAX_TEST, seed=SUBSAMPLE_SEED + 17
)

num_classes = int(len(np.unique(y_train)))
feature_dim = int(X_train.shape[1])
task = resolve_task("auto", num_classes)

radius_report = summarize_embedding_ball_radii(
    X_train,
    y_train,
    quantiles=(0.50, 0.80, 0.95),
    max_exact_pairs=250_000,
    max_sampled_pairs=100_000,
    seed=SUBSAMPLE_SEED,
)
radius = radius_value_from_report(radius_report, RADIUS_TAG)

pd.DataFrame([
    {
        "dataset": DATASET,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "feature_dim": feature_dim,
        "num_classes": num_classes,
        "task": task,
        "radius_tag": RADIUS_TAG,
        "radius": radius,
    }
])


## Theorem-backed operator-norm model

For Paper 2 we use only the dense operator-norm theorem family.  The certified record-gradient Lipschitz constant $L_z$ is computed from the public bounds $(B,A,\Lambda,H)$.


In [ ]:
B_all = float(
    max(
        np.max(np.linalg.norm(X_train, axis=1)),
        np.max(np.linalg.norm(X_test, axis=1)),
    ) * INPUT_BOUND_MARGIN
)

bounds = TheoremBounds(B=B_all, A=A, Lambda=LAMBDA)
spec = TheoremModelSpec(
    d_in=feature_dim,
    hidden_dim=HIDDEN_DIM,
    task=task,
    parameterization="dense",
    constraint="op",
    num_classes=(None if task == "binary" else num_classes),
)
constants = certified_constants(spec, bounds)
lz = certified_lz(spec, bounds)
critical_radius = 2.0 * CLIP_NORM / lz

pd.DataFrame([
    {
        "B": B_all,
        "A": A,
        "Lambda": LAMBDA,
        "H": HIDDEN_DIM,
        "L_z": lz,
        "radius": radius,
        "L_z * radius": lz * radius,
        "2C": 2.0 * CLIP_NORM,
        "Delta_ball": min(lz * radius, 2.0 * CLIP_NORM),
        "critical_radius_2C_over_Lz": critical_radius,
    }
])


## Calibrate Ball and standard DP-SGD noise

The nonconvex code uses the usual DP-SGD noise multiplier.  We calibrate it with the accountant before training.  Ball accounting uses $\min\{L_z r,2C\}$; the standard comparator uses $2C$.


In [ ]:
cal_ball = calibrate_noise(
    target_epsilon=EPSILON,
    mechanism="ball",
    n_total=len(X_train),
    num_steps=NUM_STEPS,
    batch_size=BATCH_SIZE,
    clip_norm=CLIP_NORM,
    radius=radius,
    lz=lz,
    delta=DELTA,
    orders=DEFAULT_ORDERS,
    calibration_steps=18,
)
cal_standard = calibrate_noise(
    target_epsilon=EPSILON,
    mechanism="standard",
    n_total=len(X_train),
    num_steps=NUM_STEPS,
    batch_size=BATCH_SIZE,
    clip_norm=CLIP_NORM,
    radius=radius,
    lz=lz,
    delta=DELTA,
    orders=DEFAULT_ORDERS,
    calibration_steps=18,
)

pd.DataFrame([
    {"mechanism": "ball", "noise_multiplier": cal_ball["noise_multiplier"], "achieved_epsilon": cal_ball["epsilon"]},
    {"mechanism": "standard", "noise_multiplier": cal_standard["noise_multiplier"], "achieved_epsilon": cal_standard["epsilon"]},
])


## Train Ball and standard transcript releases

Both runs use the same architecture and optimizer.  They differ only in the privacy view used to calibrate the noise multiplier.


In [ ]:
def train_demo_release(mechanism: str, noise_multiplier: float):
    model = make_model(
        spec,
        key=jr.PRNGKey(100_000 + RELEASE_SEED),
        init_project=True,
        bounds=bounds,
    )
    train_cfg = TrainConfig(
        radius=radius,
        privacy=("ball_dp" if mechanism == "ball" else "standard_dp"),
        epsilon=EPSILON,
        delta=DELTA,
        num_steps=NUM_STEPS,
        batch_size=BATCH_SIZE,
        batch_sampler="poisson",
        accountant_subsampling="match_sampler",
        clip_norm=CLIP_NORM,
        noise_multiplier=float(noise_multiplier),
        learning_rate=LEARNING_RATE,
        checkpoint_selection="last",
        eval_every=EVAL_EVERY,
        eval_batch_size=512,
        normalize_noisy_sum_by="batch_size",
        seed=RELEASE_SEED,
    )
    return fit_release(
        model,
        spec,
        bounds,
        X_train,
        y_train,
        X_eval=X_test,
        y_eval=y_test,
        train_cfg=train_cfg,
        key=jr.PRNGKey(RELEASE_SEED),
        orders=DEFAULT_ORDERS,
    )

release_ball = train_demo_release("ball", cal_ball["noise_multiplier"])
release_standard = train_demo_release("standard", cal_standard["noise_multiplier"])


## Utility and direct transcript Ball-ReRo bounds

For a uniform finite prior of size $m$, exact identification has oblivious baseline $\kappa=1/m$.  The direct Paper 2 bound is the Hayes product-reference bound.


In [ ]:
rows = []
for name, release in [("ball", release_ball), ("standard", release_standard)]:
    b = compute_hayes_bounds(release, m=M, feature_dim=feature_dim, include_rdp=True)
    rows.append({
        "mechanism": name,
        "accuracy": release.utility_metrics.get("accuracy"),
        "public_eval_loss": release.utility_metrics.get("public_eval_loss"),
        "epsilon_ball_view": release.privacy.ball.dp_certificates[0].epsilon,
        "epsilon_standard_view": release.privacy.standard.dp_certificates[0].epsilon,
        "noise_multiplier": release.training_config["noise_multipliers"][0],
        "Delta_ball_max": release.sensitivity.delta_ball,
        "Delta_std_max": release.sensitivity.delta_std,
        **b,
    })
summary = pd.DataFrame(rows)
summary


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(summary["mechanism"], summary["noise_multiplier"])
ax.set_ylabel("Calibrated noise multiplier")
ax.set_title("Ball calibration needs less noise when L_z r < 2C")
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(summary["mechanism"], summary["bound_hayes_ball"])
ax.axhline(1.0 / M, linestyle="--", label=f"oblivious baseline 1/{M}")
ax.set_ylabel("Hayes Ball-ReRo exact-ID upper bound")
ax.set_ylim(0, 1)
ax.legend()
plt.show()


## One finite-prior transcript attack

We now instantiate the attack protocol from the paper.

1. Pick a training anchor $u$.
2. Build a same-label candidate support $S\subseteq B(u,r)$ from public/test embeddings.
3. Replace $u$ in the training set by a chosen candidate target $z\in S$.
4. Record the sanitized DP-SGD transcript.
5. Subtract the known non-target batch contributions.
6. Run the exact finite-prior Bayesian KI and UI attacks.


In [ ]:
# Build support banks around feasible same-label anchors.
# The demo uses one support and one target.  The official attack script loops over many.
from quantbayes.ball_dp.experiments.run_attack_experiment import LoadedDataset

small_data = LoadedDataset(
    spec=loaded.spec,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    label_values=loaded.label_values,
    num_classes=num_classes,
    feature_dim=feature_dim,
    empirical_embedding_bound=float(np.max(np.linalg.norm(X_train, axis=1))),
    backend=loaded.backend,
)

banks = find_feasible_support_banks(
    small_data,
    radius_value=radius,
    max_required_m=M,
    num_supports=1,
    anchor_seed=0,
    source_mode="public_only",
    max_search=2000,
    explicit_anchor_indices=None,
    strict=True,
    anchor_selection="rare_class",
)
bank = banks[0]
X_support, y_support, source_ids, dists = make_support_set(
    bank,
    m=M,
    draw_index=0,
    base_seed=0,
    selection="farthest",
)

target_pos = 0
x_target = X_support[target_pos]
y_target = int(y_support[target_pos])
X_minus, y_minus = remove_train_index(small_data.X_train, small_data.y_train, bank.anchor_index)
X_full = np.concatenate([X_minus, x_target[None, :]], axis=0).astype(np.float32)
y_full = np.concatenate([y_minus, np.asarray([y_target], dtype=np.int32)], axis=0)
target_index = len(X_full) - 1
full_ds = ArrayDataset(X_full, y_full, name="demo_attack_full")
true_record = full_ds.record(target_index)

pd.DataFrame({
    "source_id": source_ids,
    "distance_to_anchor": dists,
    "is_target": [i == target_pos for i in range(M)],
})


In [ ]:
release_attack, recorder = train_recorded_release(
    spec=spec,
    bounds=bounds,
    X_train=X_full,
    y_train=y_full,
    X_eval=X_test,
    y_eval=y_test,
    mechanism="ball",
    radius=radius,
    epsilon=EPSILON,
    delta=DELTA,
    noise_multiplier=float(cal_ball["noise_multiplier"]),
    num_steps=NUM_STEPS,
    batch_size=BATCH_SIZE,
    clip_norm=CLIP_NORM,
    learning_rate=LEARNING_RATE,
    eval_every=EVAL_EVERY,
    eval_batch_size=512,
    checkpoint_selection="last",
    release_seed=RELEASE_SEED,
    orders=DEFAULT_ORDERS,
    trace_capture_every=1,
    record_operator_norms=False,
    operator_norms_every=50,
)

trace_meta = make_trace_metadata_from_release(release_attack, target_index=target_index)
trace = recorder.to_trace(
    state=release_attack.extra.get("model_state", None),
    loss_name=spec.loss_name,
    reduction=trace_meta.get("reduction", "mean"),
    metadata=trace_meta,
)
target_inclusions = sum(
    bool(np.any(np.asarray(step.batch_indices) == target_index)) for step in trace.steps
)
print({"retained_steps": len(trace.steps), "target_inclusions": target_inclusions})

residual_trace = subtract_known_batch_gradients(
    trace,
    full_ds,
    target_index=target_index,
    loss_name=spec.loss_name,
    seed=0,
)


In [ ]:
attack_rows = []
for mode in ["known_inclusion", "unknown_inclusion"]:
    status, metrics, diagnostics = run_one_attack(
        residual_trace=residual_trace,
        X_support=X_support,
        y_support=y_support,
        target_index=target_index,
        known_label=int(bank.anchor_label),
        true_record=true_record,
        mode=mode,
        attack_seed=0,
        known_inclusion_step_mode="present_steps",
    )
    attack_rows.append({
        "mode": mode,
        "status": status,
        "exact_identification_success": metrics.get("exact_identification_success"),
        "prior_rank": metrics.get("prior_rank"),
        "predicted_prior_index": diagnostics.get("predicted_prior_index"),
        "true_prior_index": diagnostics.get("true_prior_index"),
        "selected_step_count": diagnostics.get("selected_step_count"),
    })
pd.DataFrame(attack_rows)


## What to compare in the paper tables

For each dataset and configuration, the official scripts report:

- utility: public evaluation accuracy/loss;
- mechanism calibration: noise multiplier and achieved Ball/standard epsilon views;
- transcript sensitivity: $\Delta_{\mathrm{Ball}}$, $2C$, and their ratio;
- theorem bound: `ball_rero(..., mode="hayes")` at $\kappa=1/m$;
- attack success: exact identification for KI and UI finite-prior Bayesian attacks.

A run is most informative when $L_z r < 2C$.  If $L_z r \ge 2C$, Ball sensitivity saturates at the standard clipped sensitivity and the local metric policy gives no transcript-level advantage for that configuration.
